# Model validation (SN32 metrics)

Step-by-step notebook to:

1. **Load** a checkpoint (token fine-tune, DACTYL, or SN32 DeBERTa)
2. **Inspect** layer / parameter structure
3. **Evaluate** on word-level test data with SN32 scores: **fp_score**, **f1_score**, **ap_score**
4. **Try your own test sentence** (Step 6)

Metrics match `llm-detection/detection/validator/reward.py`.

In [20]:
import os
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

PROJECT_DIR = Path.cwd()

# --- change these paths to compare models ---
MODEL_DIR = PROJECT_DIR / "output" / "dactyl_token_finetuned" / "2"
WEIGHTS_PATH = None  # e.g. PROJECT_DIR / "llm-detection/models/deberta-large-ls03-ctx1024.pth"
BASE_MODEL_DIR = PROJECT_DIR / "models" / "DACTYL"

DATA_PATH = PROJECT_DIR / "pickles" / "test_samples.pkl"
MAX_SAMPLES = 500   # set None for full test set
BATCH_SIZE = 4
MAX_LENGTH = None   # None = use checkpoint default
MODEL_TYPE = "auto" # auto | token | sequence

print("MODEL_DIR:", MODEL_DIR)
print("DATA_PATH:", DATA_PATH)

MODEL_DIR: /home/ine/projects/human-ai-mixed-data-generator/output/dactyl_token_finetuned/2
DATA_PATH: /home/ine/projects/human-ai-mixed-data-generator/pickles/test_samples.pkl


## Step 1 — Load model & tokenizer

Supports:

| Path | Type |
|------|------|
| `output/dactyl_token_finetuned/1` or `2` | token (word-aligned head) |
| `models/DACTYL` | sequence (document-level, sigmoid) |
| `llm-detection/models` | sequence (SN32 foundation + `.pth`) |

In [12]:
import importlib

import torch

import validate_token_model as vtm
importlib.reload(vtm)  # pick up latest validate_token_model.py after edits

get_model_info = vtm.get_model_info
load_model_and_tokenizer = vtm.load_model_and_tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, tokenizer, default_max_length, model_kind = load_model_and_tokenizer(
    model_dir=MODEL_DIR,
    base_model_dir=BASE_MODEL_DIR,
    device=device,
    model_type=MODEL_TYPE,
    weights_path=WEIGHTS_PATH,
)

max_length = MAX_LENGTH or default_max_length
model_info = get_model_info(model)

print("device     :", device)
print("model class:", model_info["class_name"])
print("model kind :", model_kind)
print("max_length :", max_length)
print("num_labels :", model_info["num_labels"])
print("hidden_size:", model_info["hidden_size"])
print("encoder layers:", model_info["n_encoder_layers"])

device     : cuda
model class: DebertaTokenClassifier
model kind : token
max_length : 256
num_labels : 2
hidden_size: 1024
encoder layers: 24


## Step 2 — Analyse layer structure

In [13]:
def count_params(module, trainable_only=False):
    if trainable_only:
        return sum(p.numel() for p in module.parameters() if p.requires_grad)
    return sum(p.numel() for p in module.parameters())

print("=== Top-level modules ===")
for name, child in model.named_children():
    n_params = count_params(child)
    print(f"  {name:20s}  {child.__class__.__name__:40s}  {n_params:>12,} params")

total = count_params(model)
trainable = count_params(model, trainable_only=True)
print(f"\nTotal params    : {total:,}")
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.4f}%)")

if hasattr(model, "classifier"):
    clf = model.classifier
    print(f"\nClassifier head: {clf}")
    if hasattr(clf, "weight"):
        print(f"  weight shape: {tuple(clf.weight.shape)}")
        print(f"  bias shape  : {tuple(clf.bias.shape)}")

if model_info["n_encoder_layers"] is not None:
    print(
        f"\nDeBERTa encoder: {model_info['n_encoder_layers']} layers, "
        f"hidden_size={model_info['hidden_size']}"
    )

print("\n=== Full module tree (first 80 lines) ===")
tree = str(model).splitlines()
for line in tree[:80]:
    print(line)
if len(tree) > 80:
    print(f"... ({len(tree) - 80} more lines)")

=== Top-level modules ===
  deberta               DebertaV2Model                             434,012,160 params
  dropout               Dropout                                              0 params
  classifier            Linear                                           2,050 params

Total params    : 434,014,210
Trainable params: 434,014,210 (100.0000%)

Classifier head: Linear(in_features=1024, out_features=2, bias=True)
  weight shape: (2, 1024)
  bias shape  : (2,)

DeBERTa encoder: 24 layers, hidden_size=1024

=== Full module tree (first 80 lines) ===
DebertaTokenClassifier(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 1024, padding_idx=0)
      (LayerNorm): LayerNorm((1024,), eps=1e-07, elementwise_affine=True, bias=True)
      (dropout): StableDropout()
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-23): 24 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self

## Step 3 — Load evaluation data

Each sample: `{"text": str, "label": list[int]}` with one label per word (`0`=human, `1`=AI).

In [16]:
load_samples = vtm.load_samples
eval_samples = load_samples(DATA_PATH, MAX_SAMPLES)

sample = eval_samples[0]
words = sample["text"].split()
print(f"samples : {len(eval_samples)}")
print(f"example : {len(words)} words, {sum(sample['label'])} AI labels")
print("preview :", sample["text"][:200], "...")

samples : 500
example : 307 words, 122 AI labels
preview : So, does that mean there won't be a cool Alice clone army? Or was it just those select clones that were mutated. Hopefully it's the latter.

And while I was reading Isaac's monologue there, I was real ...


## Step 4 — Run inference

- **token** models: per-word P(AI) from token logits (first subword per word)
- **sequence** models: one document P(AI) replicated to every word (SN32 miner style)

In [21]:
predict_all_words = vtm.predict_all_words
predict_sequence_model = vtm.predict_sequence_model

if model_kind == "sequence":
    y_pred, y_true, stats = predict_sequence_model(
        model=model,
        samples=eval_samples,
        tokenizer=tokenizer,
        max_length=max_length,
        batch_size=BATCH_SIZE,
        device=device,
    )
else:
    y_pred, y_true, stats = predict_all_words(
        model=model,
        samples=eval_samples,
        tokenizer=tokenizer,
        max_length=max_length,
        batch_size=BATCH_SIZE,
        device=device,
        truncated_fallback="skip",
    )

print("scored words:", stats["scored_words"])
if stats.get("skipped_words"):
    print("skipped words (truncation):", stats["skipped_words"])
print("y_pred shape:", y_pred.shape)
print("P(AI) range :", float(y_pred.min()), "→", float(y_pred.max()))

Predicting: 100%|██████████| 125/125 [00:15<00:00,  8.27batch/s]

scored words: 71137
skipped words (truncation): 181941
y_pred shape: (71137,)
P(AI) range : 0.0018661344656720757 → 0.9799606204032898


## Step 5 — SN32 metrics (fp_score, f1_score, ap_score)

Same formulas as the subnet validator:

- **fp_score** = `1 - FP / N` (hard preds via `round(prob)`)
- **f1_score** = sklearn binary F1 on hard preds
- **ap_score** = sklearn average precision on raw probabilities
- **reward** = average of the three

In [22]:
sn32_reward = vtm.sn32_reward

reward, metrics = sn32_reward(y_pred, y_true)

print("=== SN32 metrics ===")
for key in ("reward", "fp_score", "f1_score", "ap_score"):
    print(f"  {key:10s}: {metrics[key]:.4f}")

print(f"\nconfusion matrix: TN={metrics['tn']} FP={metrics['fp']} FN={metrics['fn']} TP={metrics['tp']}")
print(f"words evaluated : {metrics['n_words']}")
print(f"label AI rate   : {y_true.mean():.3f}")
print(f"pred AI rate    : {(y_pred >= 0.5).mean():.3f}")

=== SN32 metrics ===
  reward    : 0.8701
  fp_score  : 0.9241
  f1_score  : 0.7953
  ap_score  : 0.8909

confusion matrix: TN=39115 FP=5396 FN=5485 TP=21141
words evaluated : 71137
label AI rate   : 0.374
pred AI rate    : 0.373


## Step 6 — Try your own test sentence

Edit `TEST_SENTENCE` below and run the cell. Works after **Step 1** (model load) — no need to run the full batch eval first.

- **token** model: different P(AI) per word
- **sequence** model: one document score applied to every word (SN32 miner style)

In [ ]:
importlib.reload(vtm)

# --- edit this sentence ---
TEST_SENTENCE = """
The quick brown fox jumps over the lazy dog. This paragraph was written by a human author
to test whether the model can detect mixed human and AI content in a single document.
Furthermore, the advanced neural networks leverage transformer architectures to generate
highly coherent synthetic text that mimics natural human writing patterns.
"""

THRESHOLD = 0.5  # P(AI) >= threshold → label "ai"

predict_text = vtm.predict_text
print_text_predictions = vtm.print_text_predictions

result = predict_text(
    model=model,
    tokenizer=tokenizer,
    text=TEST_SENTENCE,
    model_kind=model_kind,
    max_length=max_length,
    device=device,
    threshold=THRESHOLD,
)

print_text_predictions(result)

# compact colored view: human=plain, ai=marked with *
inline = []
for word, label in zip(result["words"], result["labels"]):
    if label == "ai":
        inline.append(f"[{word}*]")
    elif label == "human":
        inline.append(word)
    else:
        inline.append(f"({word}?)")
print("\ninline:", " ".join(inline[:60]) + (" ..." if len(inline) > 60 else ""))

In [ ]:
# Optional: load a real mixed sample from the test set and compare to ground truth
USE_SAMPLE_FROM_DATASET = True
SAMPLE_INDEX = 0

if USE_SAMPLE_FROM_DATASET and "eval_samples" in dir():
    ex = eval_samples[SAMPLE_INDEX]
    gt_words = ex["text"].split()
    gt_labels = ["ai" if x else "human" for x in ex["label"]]

    result_ex = predict_text(
        model=model,
        tokenizer=tokenizer,
        text=ex["text"],
        model_kind=model_kind,
        max_length=max_length,
        device=device,
        threshold=THRESHOLD,
    )

    print("=== dataset sample", SAMPLE_INDEX, "===")
    print_text_predictions(result_ex, max_words=40)

    # word-level accuracy on scored words only
    correct = 0
    scored = 0
    for pred, truth, ok in zip(result_ex["labels"], gt_labels, result_ex["scored_mask"]):
        if not ok or pred == "—":
            continue
        scored += 1
        if pred == truth:
            correct += 1
    if scored:
        print(f"\nword accuracy (scored words only): {correct}/{scored} = {correct/scored:.3f}")

## Optional — compare multiple models

Uncomment and edit paths, then run the cell.

In [23]:
compare_models = [
    PROJECT_DIR / "output/dactyl_token_finetuned/1",
    PROJECT_DIR / "output/dactyl_token_finetuned/2",
    # PROJECT_DIR / "models/DACTYL",
    # PROJECT_DIR / "llm-detection/models",
]

rows = []
for path in compare_models:
    m, tok, ml, kind = load_model_and_tokenizer(path, BASE_MODEL_DIR, device, "auto", None)
    mx = MAX_LENGTH or ml
    if kind == "sequence":
        pred, true, _ = predict_sequence_model(m, eval_samples, tok, mx, BATCH_SIZE, device)
    else:
        pred, true, _ = predict_all_words(m, eval_samples, tok, mx, BATCH_SIZE, device, "skip")
    _, met = sn32_reward(pred, true)
    rows.append({
        "model": str(path.relative_to(PROJECT_DIR)),
        "kind": kind,
        "reward": met["reward"],
        "fp_score": met["fp_score"],
        "f1_score": met["f1_score"],
        "ap_score": met["ap_score"],
    })
    del m, tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for row in rows:
    print(row)

Predicting: 100%|██████████| 125/125 [00:15<00:00,  8.18batch/s]


{'model': 'output/dactyl_token_finetuned/1', 'kind': 'token', 'reward': 0.8401785228091958, 'fp_score': 0.8930233211971267, 'f1_score': 0.763931762794476, 'ap_score': 0.8635804844359848}
{'model': 'output/dactyl_token_finetuned/2', 'kind': 'token', 'reward': 0.8701339709749405, 'fp_score': 0.9241463654638233, 'f1_score': 0.7953275774504824, 'ap_score': 0.8909279700105159}
